# 0 Imports

In [65]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd

from src.suporte import funcoes_suporte as fs

## 0.1 Funções Suporte

In [66]:
fs.jupyter_settings(altura = 10, largura = 12, fonte = 8)
fs.supressao_notacao(casa_decimal = 2)

# 0.2 Load Data

In [67]:
df_all = fs.load_pickle("../data/interim/2.all_quantity.pkl")
df_all.sample(5)

,invoice_no,stock_code,quantity,invoice_date,unit_price,country,customer_id
528077,580727,85179A,1,2011-12-05,9.13,United Kingdom,14096
18672,537823,21774,1,2010-12-08,2.51,United Kingdom,19125
321934,565210,22402,3,2011-09-01,0.39,United Kingdom,17750
495880,578333,22551,12,2011-11-24,1.65,Germany,12471
513510,579555,85071D,5,2011-11-30,0.39,United Kingdom,16241


In [68]:
df_compras = fs.load_pickle("../data/interim/2.df_compras.pkl")
df_compras.sample(5)

,invoice_no,stock_code,quantity,invoice_date,unit_price,country,customer_id
139394,548313,22403,3,2011-03-30,1.25,United Kingdom,15353
343311,566949,23201,1,2011-09-15,4.13,United Kingdom,21720
410178,572095,20718,3,2011-10-20,1.25,United Kingdom,15511
10983,537240,22796,1,2010-12-06,21.23,United Kingdom,19070
275354,560979,21984,24,2011-07-22,0.29,United Kingdom,15232


In [69]:
df_compras.columns

Index(['invoice_no', 'stock_code', 'quantity', 'invoice_date', 'unit_price',
       'country', 'customer_id'],
      dtype='object')

In [70]:
df_returns = fs.load_pickle("../data/interim/2.df_returns.pkl")
df_returns.sample(5)

,invoice_no,stock_code,quantity,invoice_date,unit_price,country,customer_id
383076,C569995,22910,-24,2011-10-06,2.95,United Kingdom,14557
281668,C561588,15056N,-2,2011-07-28,5.95,United Kingdom,15781
394051,C570867,21915,-12,2011-10-12,1.25,USA,12607
248486,C558859,22781,-1,2011-07-04,7.65,United Kingdom,17841
65098,C541693,84945,-6,2011-01-20,0.85,United Kingdom,14309


# 1.0 F.E.

Ideias extras:
1. Média Móvel -7d, 15d, 30d
2. Quantidade de compras por mês, antes do dia 15, depois do dia 15
3. Feature com foco em finanças

In [71]:
df_ref = df_compras.drop(columns = ['invoice_no', 'stock_code', 'quantity', 'invoice_date', 'unit_price', 'country']).drop_duplicates( ignore_index=True)
df_ref.head()

,customer_id
0,17850
1,13047
2,12583
3,13748
4,15100


## 1.1 Gross Revenue (Faturamento) quantidade * preço

In [72]:
df_compras["faturamento"] = df_compras["quantity"] * df_compras["unit_price"]

## 1.2 Monetário

In [73]:
df_monetario = df_compras[["customer_id", "faturamento"]].groupby("customer_id").sum().reset_index()

df_ref = pd.merge(df_ref, df_monetario, how="left", on="customer_id")

df_ref.head()

,customer_id,faturamento
0,17850,5391.21
1,13047,3232.59
2,12583,6705.38
3,13748,948.25
4,15100,876.00


In [74]:
df_ref.isna().sum()

customer_id    0
faturamento    0
dtype: int64

In [75]:
del df_monetario

## 1.3 Recência

In [76]:
df_recencia = df_compras[["customer_id", "invoice_date"]].groupby("customer_id").max().reset_index()

df_recencia["recencia_days"] = (df_compras["invoice_date"].max() - df_recencia["invoice_date"]).dt.days

df_recencia = df_recencia[["customer_id", "recencia_days"]].copy()

df_ref = pd.merge(df_ref, df_recencia, how = "left", on="customer_id")

df_ref.head()

,customer_id,faturamento,recencia_days
0,17850,5391.21,372
1,13047,3232.59,56
2,12583,6705.38,2
3,13748,948.25,95
4,15100,876.00,333


In [77]:
df_ref.isna().sum()

customer_id      0
faturamento      0
recencia_days    0
dtype: int64

In [78]:
del df_recencia

## 1.4 Qtde Compras

Atualizar no relatório, isso aqui não é frequência

In [79]:
df_qtde_compras = df_compras[["customer_id", "invoice_no"]].drop_duplicates().groupby("customer_id").count().reset_index().rename(columns = {"invoice_no": "qtde_compras"})

df_ref = pd.merge(df_ref, df_qtde_compras, how='left', on='customer_id')

df_ref.head()

,customer_id,faturamento,recencia_days,qtde_compras
0,17850,5391.21,372,34
1,13047,3232.59,56,9
2,12583,6705.38,2,15
3,13748,948.25,95,5
4,15100,876.00,333,3


In [80]:
df_ref.isna().sum()

customer_id      0
faturamento      0
recencia_days    0
qtde_compras     0
dtype: int64

In [81]:
del df_qtde_compras

## 1.5 Quantidade de produtos comprados

In [82]:
df_qtde_produto_comprado = df_compras[["customer_id", "quantity"]].drop_duplicates().groupby("customer_id").sum().reset_index().rename(columns = {"quantity": "qtde_produto_comprado"})

df_ref = pd.merge(df_ref, df_qtde_produto_comprado, how='left', on='customer_id')

df_ref.head()

,customer_id,faturamento,recencia_days,qtde_compras,qtde_produto_comprado
0,17850,5391.21,372,34,35
1,13047,3232.59,56,9,131
2,12583,6705.38,2,15,1568
3,13748,948.25,95,5,169
4,15100,876.00,333,3,48


In [83]:
df_ref.isna().sum()

customer_id              0
faturamento              0
recencia_days            0
qtde_compras             0
qtde_produto_comprado    0
dtype: int64

In [84]:
del df_qtde_produto_comprado

## 1.6 Avg Ticket

In [85]:
avg_ticket = df_compras[['customer_id','faturamento']].groupby('customer_id').mean().reset_index().rename(columns = {'faturamento':'avg_faturamento'})
df_ref = pd.merge(df_ref, avg_ticket, how='left', on='customer_id')
df_ref.head()

,customer_id,faturamento,recencia_days,qtde_compras,qtde_produto_comprado,avg_faturamento
0,17850,5391.21,372,34,35,18.15
1,13047,3232.59,56,9,131,18.90
2,12583,6705.38,2,15,1568,28.90
3,13748,948.25,95,5,169,33.87
4,15100,876.00,333,3,48,292.00


In [86]:
df_ref.isna().sum()

customer_id              0
faturamento              0
recencia_days            0
qtde_compras             0
qtde_produto_comprado    0
avg_faturamento          0
dtype: int64

## 1.7 Avg Recency Days

Podemos:
1. Excluir os NAs.
2. É uma métrica não importante para o objetivo dos insiders.

In [87]:
df_aux = df_compras[['customer_id','invoice_date']].drop_duplicates().sort_values( ['customer_id','invoice_date'], ascending=[False, False] )
df_aux['next_customer_id'] = df_aux['customer_id'].shift()
df_aux['previous_date'] = df_aux['invoice_date'].shift()

df_aux['avg_recency_days'] = df_aux.apply( lambda x: (x['previous_date'] - x['invoice_date']).days if x['customer_id'] == x['next_customer_id'] else np.nan, axis =1 )

df_aux = df_aux.drop( columns=['invoice_date','next_customer_id','previous_date']).dropna()

df_avg_recency_days = df_aux.groupby('customer_id').mean().reset_index()

df_ref = pd.merge(df_ref, df_avg_recency_days, on='customer_id', how='left')
df_ref.head(10)

,customer_id,faturamento,recencia_days,qtde_compras,qtde_produto_comprado,avg_faturamento,avg_recency_days
0,17850,5391.21,372,34,35,18.15,1.00
1,13047,3232.59,56,9,131,18.90,52.83
2,12583,6705.38,2,15,1568,28.90,26.50
3,13748,948.25,95,5,169,33.87,92.67
4,15100,876.00,333,3,48,292.00,20.00
5,15291,4623.30,25,14,508,45.33,26.77
6,14688,5630.87,7,21,579,17.22,19.26
7,17809,5411.91,16,12,961,88.72,39.67
8,15311,60767.90,0,91,2167,25.54,4.19
9,16098,2005.63,87,7,240,29.93,47.67


In [88]:
df_ref.isna().sum()

customer_id                 0
faturamento                 0
recencia_days               0
qtde_compras                0
qtde_produto_comprado       0
avg_faturamento             0
avg_recency_days         2923
dtype: int64

## 1.8 Compras Frequency 

In [89]:
df_aux = df_compras[['customer_id', 'invoice_no', 'invoice_date']].drop_duplicates()\
                                                                    .groupby('customer_id')\
                                                                    .agg( max_ = ('invoice_date','max'),
                                                                          min_ = ('invoice_date','min'),
                                                                          days_= ('invoice_date', lambda x: (x.max() - x.min()).days +1),
                                                                          buy_ = ('invoice_no', 'count')
                                                                         ).reset_index()

df_aux['frequencia'] = df_aux[['buy_','days_']].apply( lambda x: x['buy_'] / x['days_'] if x['days_'] !=0 else 0, axis =1)

df_ref = pd.merge(df_ref, df_aux[['customer_id','frequencia']], on='customer_id', how='left')

df_ref.head(5)

,customer_id,faturamento,recencia_days,qtde_compras,qtde_produto_comprado,avg_faturamento,avg_recency_days,frequencia
0,17850,5391.21,372,34,35,18.15,1.00,17.00
1,13047,3232.59,56,9,131,18.90,52.83,0.03
2,12583,6705.38,2,15,1568,28.90,26.50,0.04
3,13748,948.25,95,5,169,33.87,92.67,0.02
4,15100,876.00,333,3,48,292.00,20.00,0.07


In [90]:
df_ref.isna().sum()

customer_id                 0
faturamento                 0
recencia_days               0
qtde_compras                0
qtde_produto_comprado       0
avg_faturamento             0
avg_recency_days         2923
frequencia                  0
dtype: int64

## 1.8 Devoluções

In [91]:
df_ret = df_returns[['customer_id', 'invoice_no']].drop_duplicates().groupby('customer_id').count().reset_index().rename(columns ={'invoice_no': 'retornos'})
df_ref = pd.merge(df_ref, df_ret, how='left', on='customer_id')
df_ref.loc[df_ref['retornos'].isna(),'retornos'] = 0
df_ref.head()

,customer_id,faturamento,recencia_days,qtde_compras,qtde_produto_comprado,avg_faturamento,avg_recency_days,frequencia,retornos
0,17850,5391.21,372,34,35,18.15,1.00,17.00,1.00
1,13047,3232.59,56,9,131,18.90,52.83,0.03,7.00
2,12583,6705.38,2,15,1568,28.90,26.50,0.04,2.00
3,13748,948.25,95,5,169,33.87,92.67,0.02,0.00
4,15100,876.00,333,3,48,292.00,20.00,0.07,3.00


In [92]:
df_ref.isna().sum()

customer_id                 0
faturamento                 0
recencia_days               0
qtde_compras                0
qtde_produto_comprado       0
avg_faturamento             0
avg_recency_days         2923
frequencia                  0
retornos                    0
dtype: int64

## 1.9 Retirar NA

In [94]:
df_ref = df_ref.dropna()
df_ref.isna().sum()

customer_id              0
faturamento              0
recencia_days            0
qtde_compras             0
qtde_produto_comprado    0
avg_faturamento          0
avg_recency_days         0
frequencia               0
retornos                 0
dtype: int64

# 2.0 Exportar DF

In [97]:
path = "../data/interim/3.fe.pkl"
fs.save_pickle(obj=df_ref,path=path)